# Device Log Generation

## Overview
Generates synthetic device telemetry data and streams it to Azure Event Hub for real-time ingestion testing.

**Purpose**: Simulate telecom device logs with realistic patterns and intentional data quality issues for testing data pipelines.

---

In [0]:
import json
import random
import time
from datetime import datetime, timedelta
from azure.eventhub import EventHubProducerClient, EventData


## Event Hub Configuration
Azure Event Hub connection setup for streaming device telemetry.



In [0]:

CONNECTION_STR = "****************************"
EVENT_HUB_NAME = "eventhubtopic"

producer = EventHubProducerClient.from_connection_string(
    conn_str=CONNECTION_STR,
    eventhub_name=EVENT_HUB_NAME
)


---
## Master Data
Reference lists for device types, regions, and health statuses used in event generation.


In [0]:

device_types = [
    "router",
    "server",
    "switch",
    "firewall",
    "load_balancer"
]

regions = [
    "Mumbai",
    "Pune",
    "Delhi",
    "Bangalore",
    "Hyderabad",
    "Chennai"
]

statuses = [
    "healthy",
    "warning",
    "critical"
]


---
## Event Generator
Function to generate synthetic device telemetry with random metrics (CPU, memory, latency, packet loss, temperature).


In [0]:

def generate_event():

    device_id = f"DVC_{random.randint(100,999)}"

    event = {
        "device_id": device_id,
        "device_type": random.choice(device_types),
        "region": random.choice(regions),
        "cpu_usage": round(random.uniform(10, 98), 2),
        "memory_usage": round(random.uniform(15, 97), 2),
        "packet_loss": random.randint(0, 10),
        "latency_ms": random.randint(5, 500),
        "temperature": round(random.uniform(30, 95), 2),
        "status": random.choice(statuses),
        "event_time": datetime.utcnow().isoformat()
    }



---
## Dirty Data Simulation
Inject data quality issues: missing values, invalid ranges, late-arriving events, schema drift.


In [0]:

    dirty_case = random.randint(1, 10)

    # Missing cpu_usage
    if dirty_case == 1:
        event["cpu_usage"] = None

    # Invalid cpu value
    elif dirty_case == 2:
        event["cpu_usage"] = 250

    # Late arriving event
    elif dirty_case == 3:
        late_time = datetime.utcnow() - timedelta(hours=5)
        event["event_time"] = late_time.isoformat()

    # Schema drift
    elif dirty_case == 4:
        event["temp"] = event.pop("temperature")

    return event


---
## Stream Events
Continuous loop sending generated events to Event Hub every 2 seconds.


In [0]:


while True:

    try:
        event = generate_event()

        event_json = json.dumps(event)

        batch = producer.create_batch()
        batch.add(EventData(event_json))

        producer.send_batch(batch)

        print(f"Sent Event: {event_json}")

        time.sleep(2)

    except Exception as e:
        print("Error:", e)

---
## Summary

**Purpose**: Synthetic telemetry data generator for testing real-time ingestion pipelines.

**Event Schema**:
* device_id, device_type, region
* cpu_usage, memory_usage, temperature
* packet_loss, latency_ms
* status (healthy/warning/critical)
* event_time (ISO timestamp)

**Data Quality Issues**:
* 10% chance of dirty data (missing values, invalid ranges, late arrivals, schema drift)

**Stream Target**: Azure Event Hub (eventhubtopic)

**Rate**: 1 event every 2 seconds